# Prompt Engineering

In [ ]:
from openai import OpenAI

# client = OpenAI(
#     base_url="https://your-api-proxy.example.com/v1",
#     api_key="sk-..."
# )
# model = "gpt-3.5-turbo"

client = OpenAI(
    base_url="http://localhost:11434/v1/",
    api_key="ollama"
)
model = "qwen2:7b"

In [5]:
def get_completion(prompt, model=model):
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0
    )
    return response.choices[0].message.content

In [13]:
## Principle1: Write clear and specific instructions

Hello! How can I assist you today?


In [6]:
### Strategy1: Use delimiters to clearly indicate distinct parts of the input
text = f"""
You should express what you want a model to do by \
providing instructions that are as clear and \
specific as you can possibly make them. \
This will guide the model towards the desired output, \
and reduce the chances of receiving irrelevant \
or incorrect responses. Don't confuse writing a \
clear prompt with writing a short prompt. \
In many cases, longer prompts provide more clarity \
and context for the model, which can lead to \
more detailed and relevant outputs.
"""
prompt = f"""
Summarize the text delimited by triple backticks \
into a single sentence.
```{text}```
"""
response = get_completion(prompt)
print(response)

The text advises clearly and specifically instructing a model on what you want it to do by crafting unambiguous directions that may be lengthy but offer ample context, thus guiding the output towards the desired result while minimizing irrelevant or incorrect responses.


In [7]:
### Strategy2: Ask for structured output:
# - JSON
# - HTML
prompt = f"""
Generate a list of three made-up book titles along \
with their authors and genres.  \
Provide them in JSON format with the following keys: \
book_id, title, author, genre.
"""
response = get_completion(prompt)
print(response)

[
{
"book_id": "1",
"title": "The Enchanted Lighthouse",
"author": "Evelyn Starlight",
"genre": "Fantasy"
},
{
"book_id": "2",
"title": "Mysteries of the Deep Sea",
"author": "Alexander Triton",
"genre": "Science Fiction"
},
{
"book_id": "3",
"title": "The Whispering Shadows",
"author": "Lena Nightshade",
"genre": "Horror"
}
]


In [39]:
### Strategy3: Ask the model to check whether conditions are satisfied
text = f"""
Making a cup of tea is easy! First, you need to get some \
water boiling. While that's happening, \
grab a cup and put a tea bag in it. Once the water is \
hot enough, just pour it over the tea bag. \
Let it sit for a bit so the tea can steep. After a \
few minutes, take out the tea bag. If you \
like, you can add some sugar or milk to taste. \
And that's it! You've got yourself a delicious \
cup of tea to enjoy.
"""
prompt = f"""
You will be provided with text delimited by triple quotes.
If it contains a sequence of instructions. \
re-write those instructions in the following format:

Step 1 - ...
Step 2 - ...
...
Step N - ...

If the text doesn't contain a sequence of instructions, \
then simply write \"No steps provide.\"
"\"\"{text}\"\"\"
"""
response = get_completion(prompt)
print(response)

Step 1 - 准备一些水，让它沸腾。
Step 2 - 在等待水沸腾的同时，拿起一个杯子并放入一个茶包。
Step 3 - 水烧开后，将热水倒入装有茶包的杯中。
Step 4 - 让茶浸泡一会儿，让其充分释放味道。
Step 5 - 等几分钟后，取出茶包。如果喜欢，可以按口味添加糖或牛奶。
Step 6 - 完成！享受你自制的美味红茶吧。


In [41]:
### Strategy4: Few-shot prompting
prompt = f"""
You task is to answer in a consistent style in chinese.

<child>: Teach me about patience.

<grandparent>: The river that carves the deepest \
valley flows from a modest spring; the \
grandest symphony originates from a single note; \
the most intricate tapestry begins with a solitary thread.

<child>: Teach me about resilience.
"""
response = get_completion(prompt)
print(response)

<grandparent>: 就像一棵树在暴风雨中摇摆，但最终仍能挺立不倒，坚韧是生命的力量。面对挫折和困难时，我们可能会感到困惑或沮丧，但正是这些挑战塑造了我们的意志，使我们变得更强大。就像橡树的根深深扎入土壤，坚韧的人会在逆境中找到成长的机会，学习从失败中汲取教训，并继续前进。

在生活旅程中，每个人都会遇到风浪和坎坷。关键在于如何面对它们：是选择放弃，还是勇敢地站起来，用行动证明自己的价值？坚韧不拔的精神让我们能够坚持到最后，即使道路崎岖，也能看到希望的光芒。就像一只蝴蝶破茧而出，经历痛苦的过程后，我们才能展现出真正的美丽与力量。

所以，孩子，在你遇到挑战时，请记得，坚韧是你的盔甲，它将帮助你在生活的海洋中航行，无论风浪多么汹涌。


In [ ]:
## Principle2: Give the model time to "think"

In [43]:
### Strategy1: Specify the steps required to complete a task
prompt = f"""
Your task is to perform the following actions:  \
1 - Summarize the following text delimited by <> with 1 sentence. \
2 - Translate the summary to Chinese. \

Text: <{text}>
"""
response = get_completion(prompt)
print(response)

1 - 水を沸かす、カップとティーバッグを入れる、水をティーバッグに注ぐ、少しついらせて浸し、好みで砂糖やミルクを加えて美味しく飲む- これらがティーコンサートの簡単な方法です。

2 - 1つのカップのティーを作るのは簡単だ！まず、水を沸かす。それを見守りながら、カップにティーバッグを入れる。水が十分に熱くなったら、それをティーバッグに注ぐ。少しついらせて浸し、好みで砂糖やミルクを加えて美味しく飲む- これがティーコンサートの簡単な方法だ。


In [44]:
### Strategy2: Instruct the model to work out its own solution before rushing to a conclusion
prompt = f"""
Your task is to determine if the student's solution \
is correct or not.
To solve the problem do the following:
- First, work out your own solution to the problem.
- Then compare your solution to the student's solution \
and evaluate if the student's solution is correct or not.
Don't decide if the student's solution is correct until
you have done the problem yourself.

Use the following format:
Question:
```
question here
```
Student's solution:
```
student's solution here
```
Actual solution:
```
steps to work out the solution and your solution here
```
Is the student's solution the same as actual solution \
just calculated:
```
yes or no
```
Student grade:
```
correct or incorrect
```

Question:
```
I'm building a solar power installation and I need help \
working out the financials.
- Land costs $100 / square foot
- I can buy solar panels for $250 / square foot
- I negotiated a contract for maintenance that will cost \
me a flat $100k per year, and an additional $10 / square \
foot
What is the total cost for the first year of operations \
as a function of the number of square feet.
```
Student's solution:
```
Let x be the size of the installation in square feet.
Costs:
1. Land cost: 100x
2. Solar panel cost: 250x
3. Maintenance cost: 100,000 + 100x
Total cost: 100x + 250x + 100,000 + 100x = 450x + 100,000
```
Actual solution:
"""
response = get_completion(prompt)
print(response)

Let x be the size of the installation in square feet.

Costs:

1. Land cost: \(100 \times x\) (since land costs $100 per square foot)
2. Solar panel cost: \(250 \times x\) (since solar panels cost $250 per square foot)
3. Maintenance cost:
   - Flat annual fee: $100,000
   - Additional fee per square foot: \(10 \times x\)

Total cost for the first year of operations:

\[ \text{Total cost} = 100x + 250x + 100,000 + 10x \]

Combine like terms:

\[ \text{Total cost} = 360x + 100,000 \]

Is the student's solution the same as actual solution just calculated:
```
no
```

Student grade:
```
incorrect
```


In [ ]:
## Iterative Prompt Development
# 1. Try something, be clear and specific
# 2. Analyze why result does not given desired output
# 3. Clarify instruction, given more time to think
# 4. Refine the idea and the prompt with a batch of examples
# 5. Repeat

## Refs
1. [Notes on ChatGPT Prompt Engineering Course](https://aaronnotes.com/2023/04/notes-on-chatgpt-prompt-engineering-course/)